Importing Libraries and load data

In [99]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split , cross_val_score
from sklearn.preprocessing import OneHotEncoder , OrdinalEncoder , TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

df = pd.read_csv("telecom_churn.csv")


In [60]:
print (df.shape)
print(df.dtypes)
df.head()

(2000, 10)
customer_id            str
tenure_months        int64
age                  int64
monthly_charges    float64
total_charges      float64
contract_type          str
payment_method         str
region                 str
signup_date            str
churn                int64
dtype: object


,customer_id,tenure_months,age,monthly_charges,total_charges,contract_type,payment_method,region,signup_date,churn
0,CUST00001,22,40,79.58,1751.54,One year,Mailed check,West,2022-08-03,1
1,CUST00002,72,30,96.73,7345.50,Month-to-month,Credit card,East,2022-08-08,1
2,CUST00003,60,43,87.25,5078.43,Two year,Electronic check,North,2023-03-17,0
3,CUST00004,20,25,97.50,1924.71,One year,Mailed check,West,2021-10-12,0
4,CUST00005,1,27,62.24,62.28,Month-to-month,Credit card,North,2023-06-09,1


In [61]:
# split data

X = df.drop(columns=["customer_id","churn"])
y = df["churn"]
X_train , X_test , y_train , y_test = train_test_split(X,y , test_size=0.2 , random_state=42 , stratify=y)

# missing value fill up 

raw_cols = ["tenure_months", "age", "monthly_charges", "total_charges"]
X_train_raw = X_train[raw_cols].fillna(0)
X_test_raw = X_test[raw_cols].fillna(0)




TASK 01 — One-hot, ordinal, and target encoding side by side

• Identify the categorical columns in telecom_churn.csv (contract_type, payment_method, region).
Separate them from the date and the id columns.


In [62]:
categorical_columns = df.select_dtypes(
    include="object"
).columns.tolist()

categorical_columns = [
    col for col in categorical_columns
    if col not in ["signup_date", "customer_id"]
]

print(categorical_columns)

['contract_type', 'payment_method', 'region']


C:\Users\morik\AppData\Local\Temp\ipykernel_31720\2449238853.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


• Apply each encoding to the categorical columns, each fit on X_train only: one-hot, ordinal, and target
(mean) encoding

In [63]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
payment_onehot = ohe.fit_transform(X_train[["payment_method"]])
print("payment_method categories:", ohe.categories_[0])
print("one-hot shape:", payment_onehot.shape, "(one column per category)")
print(pd.DataFrame(payment_onehot, columns=ohe.get_feature_names_out()).head(3))

contract_onehot = ohe.fit_transform(X_train[["contract_type"]])
print("Contract_method categories:", ohe.categories_[0])
print("one-hot shape:", contract_onehot.shape, "(one column per category)")
print(pd.DataFrame(contract_onehot, columns=ohe.get_feature_names_out()).head(3))

region_onehot = ohe.fit_transform(X_train[["region"]])
print("region based categories:", ohe.categories_[0])
print("one-hot shape:", region_onehot.shape, "(one column per category)")
print(pd.DataFrame(region_onehot, columns=ohe.get_feature_names_out()).head(3))


X_train_onehot_final = np.hstack([
    X_train_numeric,
    payment_onehot,
    contract_onehot,
    region_onehot
])

print("One-Hot final shape:", X_train_onehot_final.shape)

payment_method categories: ['Bank transfer' 'Credit card' 'Electronic check' 'Mailed check']
one-hot shape: (1600, 4) (one column per category)
   payment_method_Bank transfer  ...  payment_method_Mailed check
0                           0.0  ...                          0.0
1                           0.0  ...                          0.0
2                           0.0  ...                          0.0

[3 rows x 4 columns]
Contract_method categories: ['Month-to-month' 'One year' 'Two year']
one-hot shape: (1600, 3) (one column per category)
   contract_type_Month-to-month  contract_type_One year  contract_type_Two year
0                           1.0                     0.0                     0.0
1                           1.0                     0.0                     0.0
2                           1.0                     0.0                     0.0
region based categories: ['Central' 'East' 'North' 'South' 'West']
one-hot shape: (1600, 5) (one column per category)
   region_Ce

In [64]:
contract_order = [["Month-to-month", "One year", "Two year"]]
oe = OrdinalEncoder(categories=contract_order)
contract_ordinal = oe.fit_transform(X_train[["contract_type"]])
print("contract_type -> ordinal code (first 5):")
print(X_train["contract_type"].values[:5])
print(contract_ordinal[:5].ravel())

payment_order = [['Bank transfer', 'Credit card', 'Electronic check' ,'Mailed check']]
oe = OrdinalEncoder(categories=payment_order)
payment_ordinal = oe.fit_transform(X_train[["payment_method"]])
print("payment_type -> ordinal code (first 5):")
print(X_train["payment_method"].values[:5])
print(payment_ordinal[:5].ravel())

region_order = [['Central' , 'East' , 'North' , 'South' , 'West']]
oe = OrdinalEncoder(categories=region_order)
region_ordinal = oe.fit_transform(X_train[["region"]])
print("region_type -> ordinal code (first 5):")
print(X_train["region"].values[:5])
print(region_ordinal[:5].ravel())

X_train_ordinal_final = np.hstack([
    X_train_numeric,
    payment_ordinal,
    contract_ordinal,
    region_ordinal
])



contract_type -> ordinal code (first 5):
<StringArray>
['Month-to-month', 'Month-to-month', 'Month-to-month',       'One year',
 'Month-to-month']
Length: 5, dtype: str
[0. 0. 0. 1. 0.]
payment_type -> ordinal code (first 5):
<StringArray>
[     'Credit card',      'Credit card', 'Electronic check',
     'Mailed check',     'Mailed check']
Length: 5, dtype: str
[1. 1. 2. 3. 3.]
region_type -> ordinal code (first 5):
<StringArray>
['East', 'North', 'West', 'North', 'South']
Length: 5, dtype: str
[1. 2. 4. 2. 3.]


In [65]:
te = TargetEncoder(target_type="binary")

payment_target = te.fit_transform(X_train[["payment_method"]], y_train)
payment_summary = pd.DataFrame({
    "payment_method": X_train["payment_method"].values,
    "encoded": payment_target.ravel()
}).drop_duplicates().sort_values("payment_method")
print("\nPayment Summary\n",payment_summary)


contract_target = te.fit_transform(X_train[["contract_type"]], y_train)
contract_summary = pd.DataFrame({
    "contract_type": X_train["contract_type"].values,
    "encoded": contract_target.ravel()
}).drop_duplicates().sort_values("contract_type")
print("\n Contract Summerry\n",contract_summary)


region_target = te.fit_transform(X_train[["region"]], y_train)
region_summary = pd.DataFrame({
    "region": X_train["region"].values,
    "encoded": region_target.ravel()
}).drop_duplicates().sort_values("region")

X_train_target_final = np.hstack([
    X_train_numeric,
    payment_target,
    contract_target,
    region_target
])

print("\n Region Summerry\n",region_summary)


Payment Summary
       payment_method   encoded
30     Bank transfer  0.181947
5      Bank transfer  0.170570
6      Bank transfer  0.183135
19     Bank transfer  0.184835
14     Bank transfer  0.175519
0        Credit card  0.185555
1        Credit card  0.182236
33       Credit card  0.195139
25       Credit card  0.186015
9        Credit card  0.195065
18  Electronic check  0.318679
11  Electronic check  0.309300
40  Electronic check  0.310036
2   Electronic check  0.300049
12  Electronic check  0.325020
8       Mailed check  0.236397
21      Mailed check  0.235277
4       Mailed check  0.228918
3       Mailed check  0.224564
41      Mailed check  0.239364

 Contract Summerry
      contract_type   encoded
0   Month-to-month  0.340740
1   Month-to-month  0.331227
7   Month-to-month  0.334586
11  Month-to-month  0.330303
26  Month-to-month  0.332678
3         One year  0.119703
5         One year  0.118380
12        One year  0.117668
20        One year  0.127409
24        One year  

• For each encoding, train a LogisticRegression on the scaled numeric features and report
cross-validation score on the training set.

In [66]:
numeric_columns = [
    "tenure_months",
    "age",
    "monthly_charges",
    "total_charges"
]


# Fill missing values using training median
for col in numeric_columns:
    median_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)


scaler = StandardScaler()

X_train_numeric = scaler.fit_transform(
    X_train[numeric_columns]
)

X_test_numeric = scaler.transform(
    X_test[numeric_columns]
)

print("Scaled numeric shape:", X_train_numeric.shape)


model_onehot = LogisticRegression(max_iter=1000)

onehot_scores = cross_val_score(
    model_onehot,
    X_train_onehot_final,
    y_train,
    cv=5,
    scoring="accuracy"
)

model_ordinal = LogisticRegression(max_iter=1000)

ordinal_scores = cross_val_score(
    model_ordinal,
    X_train_ordinal_final,
    y_train,
    cv=5,
    scoring="accuracy"
)
model_target = LogisticRegression(max_iter=1000)

target_scores = cross_val_score(
    model_target,
    X_train_target_final,
    y_train,
    cv=5,
    scoring="accuracy"
)



print("One-Hot CV scores:", onehot_scores)
print("One-Hot Mean CV:", onehot_scores.mean())

print("\nOrdinal CV scores:", ordinal_scores)
print("Ordinal Mean CV:", ordinal_scores.mean())

print("\nTarget Mean CV scores:", target_scores)
print("Target Mean CV:", target_scores.mean())


Scaled numeric shape: (1600, 4)
One-Hot CV scores: [0.778125 0.784375 0.753125 0.7625   0.784375]
One-Hot Mean CV: 0.7725

Ordinal CV scores: [0.771875 0.775    0.76875  0.7875   0.784375]
Ordinal Mean CV: 0.7775000000000001

Target Mean CV scores: [0.76875  0.771875 0.771875 0.778125 0.778125]
Target Mean CV: 0.77375


**Write one sentence per encoding: when would this be the right choice, and why do the CV scores
differ?**

One-Hot: Best for categories with no natural order.

Ordinal: Best when categories have a meaningful order.

Target Mean: Best for high-cardinality categories using target relationship.

CV scores differ: Different encodings represent categorical data differently, affecting model performance.

****TASK 02 — Scaling: standard, min-max, and robust****

• On the numeric features, apply StandardScaler, MinMaxScaler, and RobustScaler — each fit on
X_train only.

• Look at monthly_charges and total_charges specifically. Which numbers look like outliers, and which
scaler is designed to survive them?

• Fit the same logistic or Ridge model under each scaler and compare cross-validation scores.

• Write one sentence on when RobustScaler beats the other two.

In [67]:
#For Monthly Charges 

demo_values = X_train["monthly_charges"].copy()
demo_with_outlier = pd.concat([demo_values, pd.Series([500.0])], ignore_index=True)  # a wild outlier

for name, scaler in [("StandardScaler", StandardScaler()),
                      ("MinMaxScaler", MinMaxScaler()),
                      ("RobustScaler", RobustScaler())]:
    scaled = scaler.fit_transform(demo_with_outlier.values.reshape(-1, 1)).ravel()
    normal_range = (scaled[:-1].min(), scaled[:-1].max())  # everyone except the outlier
    outlier_value = scaled[-1]
    print(f"{name:15}: normal customers land in [{normal_range[0]:.2f}, {normal_range[1]:.2f}], "
          f"the outlier lands at {outlier_value:.2f}")

StandardScaler : normal customers land in [-1.73, 3.16], the outlier lands at 16.12
MinMaxScaler   : normal customers land in [0.00, 0.27], the outlier lands at 1.00
RobustScaler   : normal customers land in [-1.34, 2.55], the outlier lands at 12.88


In [68]:
#For Total Charges

total_demo_values = X_train["total_charges"].copy()

total_demo_outlier = pd.concat(
    [total_demo_values, pd.Series([10000.0])],
    ignore_index=True
)

for name, scaler in [
    ("StandardScaler", StandardScaler()),
    ("MinMaxScaler", MinMaxScaler()),
    ("RobustScaler", RobustScaler())
]:
    total_scaled = scaler.fit_transform(
        total_demo_outlier.values.reshape(-1, 1)
    ).ravel()

    total_normal_range = (
        total_scaled[:-1].min(),
        total_scaled[:-1].max()
    )

    total_outlier_value = total_scaled[-1]

    print(
        f"{name:15}: normal customers land in "
        f"[{total_normal_range[0]:.2f}, {total_normal_range[1]:.2f}], "
        f"the outlier lands at {total_outlier_value:.2f}"
    )

StandardScaler : normal customers land in [-0.92, 4.59], the outlier lands at 5.70
MinMaxScaler   : normal customers land in [0.00, 0.83], the outlier lands at 1.00
RobustScaler   : normal customers land in [-0.55, 4.68], the outlier lands at 5.73


TASK 03 — Polynomial and interaction features that earn their place

• Add polynomial features (degree 2) to a sensible subset of the numeric features using
PolynomialFeatures

In [69]:
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(X_train[["tenure_months","monthly_charges"]])

feature_names = poly.get_feature_names_out(["tenure_months","monthly_charges"])
print("Feature names:", feature_names)

Feature names: ['tenure_months' 'monthly_charges' 'tenure_months^2'
 'tenure_months monthly_charges' 'monthly_charges^2']


• Compare, in one small table, the cross-validation score with and without the polynomial features for
the same model.

In [70]:
model = LinearRegression()
score_original = cross_val_score(
    model,
    X_train[["tenure_months","monthly_charges"]],
    y_train,
    cv=5,
    scoring = "r2"
)

score_poly = cross_val_score(
    model,
    poly_features,
    y_train,
    cv=5,
    scoring = "r2"
)

comparision = pd.DataFrame({
    "Original": score_original,
    "Polynomial": score_poly
})

print("Comparision of Original and Polynomial Features:\n",comparision)

Comparision of Original and Polynomial Features:
    Original  Polynomial
0  0.033184    0.031991
1  0.049828    0.050502
2  0.049647    0.052489
3  0.038030    0.036555
4  0.073656    0.069134


• Report whether the extra features actually helped, and by how much. If some interaction is clearly
strong, name it.

In [71]:
poly_model = LinearRegression()
poly_model.fit(poly_features, y_train)

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": poly_model.coef_
})

print(coef_df.sort_values("Coefficient", key=abs, ascending=False))

                         Feature  Coefficient
1                monthly_charges     0.005427
0                  tenure_months    -0.001347
2                tenure_months^2    -0.000020
4              monthly_charges^2    -0.000016
3  tenure_months monthly_charges    -0.000007


Ans :- The extra polynomial features did not help No interaction was clearly strong; the tenure_months × monthly_charges interaction had a very small coefficient of -0.000007.

• Rerun the better one under Ridge and note whether the extra terms now behave themselves.

In [72]:
ridge_original = Ridge(alpha=1.0)
ridge_poly = Ridge(alpha=1.0)

original_ridge_scores = cross_val_score(
    ridge_original,
    X_train[["tenure_months", "monthly_charges"]],
    y_train,
    cv=5,
    scoring="r2"
)

poly_ridge_scores = cross_val_score(
    ridge_poly,
    poly_features,
    y_train,
    cv=5,
    scoring="r2"
)

print("Original Ridge:", original_ridge_scores.mean())
print("Polynomial Ridge:", poly_ridge_scores.mean())

Original Ridge: 0.04886914284507893
Polynomial Ridge: 0.048134234649817184


Answer:- Original features performed better (CV R² = 0.048869). Ridge gave the same score, so the extra polynomial terms did not improve the model

TASK 04 — Turn the date column into usable features

• Parse signup_date into a real pandas datetime and confirm you can read off year, month,
day-of-week, and quarter.

• Engineer at least three derived numeric features from the date (for example year, month, and days
since the latest signup date).

• Add the best date features to your feature set and report whether cross-validation score improves.

• Drop the raw date string afterward — a raw string is not a feature.

In [92]:
dates = pd.to_datetime(X_train["signup_date"])

df["signup_month"] = dates.dt.month
df["signup_year"] = dates.dt.year
df["signup_dayofweek"] = dates.dt.dayofweek
df["days_since_signup"] =(dates.max() - dates).dt.days


In [74]:
#Train the date data for date features
date_features = [
    "signup_month",
    "signup_year",
    "signup_dayofweek",
    "days_since_signup"
]

X_train_date = X_train.copy()

X_train_date[date_features] = df[date_features]

# Drop raw date string
X_train_date = X_train_date.drop(columns=["signup_date"])

In [75]:
model = LinearRegression()

score_original = cross_val_score(
    model,
    X_train[["tenure_months", "monthly_charges"]],
    y_train,
    cv=5,
    scoring="r2"
).mean()

score_date = cross_val_score(
    model,
    X_train_date[[
        "tenure_months",
        "monthly_charges",
        "signup_month",
        "signup_year",
        "signup_dayofweek",
        "days_since_signup"
    ]],
    y_train,
    cv=5,
    scoring="r2"
).mean()

comparison = pd.DataFrame({
    "Feature Set": ["Original", "With Date Features"],
    "Mean CV R²": [score_original, score_date]
})

print(comparison)

          Feature Set  Mean CV R²
0            Original    0.048869
1  With Date Features    0.049610


TASK 05 — Spot the leakage in this code, then fix it

In [76]:

X_numeric = X.select_dtypes(include="number")

scaler = StandardScaler()

# WRONG: leakage demonstration
X_all_scaled = scaler.fit_transform(X_numeric)

X_train_s, X_test_s = train_test_split(
    X_all_scaled,
    test_size=0.2,
    random_state=42
)

• Name the leak precisely: which statistics of the test rows were seen by the training transform?

==> The training transform has seen the mean and standard deviation of every feature calculated using both training and test rows.

• Explain the consequence in one paragraph — what does the training transform now 'know' that a real
deployment would never give it?

==> > Fitting the scaler on the test set causes **data leakage** because the scaler learns information from unseen data. This can make the model’s evaluation slightly optimistic and less representative of real-world performance.


• Rewrite it so every fit happens on training folds only, inside a Pipeline / ColumnTransformer.



In [77]:
numeric_features = X_train.select_dtypes(include="number").columns
categorical_features = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])



Prove the fix: show the cross-validation score of the leaky version vs the fixed version, and explain
any gap.

In [78]:
cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

# 6. Results
print("CV Scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())

CV Scores: [-0.33734658 -0.38651745 -0.52078107 -0.36885394 -0.34773521]
Mean CV R²: -0.3922468507551774


TASK 06 — Assemble the full ColumnTransformer + Pipeline

• Build a ColumnTransformer that routes: numeric columns to scaling, categorical columns to one-hot
encoding, and the date features to an appropriate transform.

• Wrap the ColumnTransformer in a Pipeline with your chosen classifier.

• Run cross-validation on the whole Pipeline and make sure it beats your best single-task score from
Part A or B.

• This ColumnTransformer is the deliverable core — it should handle the full DataFrame in one fit.

In [97]:
X = X.copy()

X["signup_month"] = df["signup_month"]
X["signup_year"] = df["signup_year"]
X["signup_dayofweek"] = df["signup_dayofweek"]
X["days_since_signup"] = df["days_since_signup"]



numeric_cols = [
  "tenure_months",
    "age",
    "monthly_charges",
    "total_charges"
]

# Categorical columns
categorical_cols = [
    "contract_type",
    "payment_method",
     "region"
]

# Date-derived numeric columns
date_cols = [
   "signup_year",
    "signup_month",
    "signup_dayofweek",
    "days_since_signup"
]

# ColumnTransformer
prep = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_cols
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_cols
    ),
    (
        "date",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        date_cols
    )
])
print(prep)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure_months', 'age', 'monthly_charges',
                                  'total_charges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['contract_type', 'payment_method', 'region']),
                                ('date',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy=

In [98]:
pipe = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=1000))
])

cv_scores = cross_val_score(
    pipe,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("CV Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

CV Scores: [0.7775 0.74   0.76   0.7775 0.78  ]
Mean CV Accuracy: 0.767


• Write a custom transformer holding onto BaseEstimator and TransformerMixin, implementing fit and
transform.

• Make it do something genuinely useful: a custom date-feature extractor, an outlier capper, a
hand-rolled interaction, or a target-aware encoder.

• Insert it into your ColumnTransformer or Pipeline and show it parcels through cross-validation without
error.

• Confirm it does not leak: anything it learns must be fitted on the training folds only.

In [100]:
class OutlierCap(BaseEstimator, TransformerMixin):

    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        # Learn limits ONLY from training data
        X = pd.DataFrame(X)

        self.lower_caps_ = X.quantile(self.lower)
        self.upper_caps_ = X.quantile(self.upper)

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        # Apply limits learned during fit()
        X = X.clip(
            lower=self.lower_caps_,
            upper=self.upper_caps_,
            axis=1
        )

        return X


In [101]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier_cap", OutlierCap()),
    ("scaler", StandardScaler())
])

In [102]:
cv_scores = cross_val_score(
    pipe,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("CV Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

CV Scores: [0.7775 0.74   0.76   0.7775 0.78  ]
Mean CV Accuracy: 0.767


Conclusion  ==> Result: The CV accuracy remained the same after applying the custom OutlierCap transformer. This indicates that outliers had little impact on the model's performance. The custom transformer was successfully integrated without data leakage.